# 03 - Spectral Indices for Agriculture & Soil Analysis

**Arab Youth Space Hackathon 2026** | Team: ليوا الخضراء | Theme: Precision Agriculture & Crop Intelligence

This notebook computes multispectral and hyperspectral indices for:
- Vegetation health (NDVI, EVI, SAVI, NDRE, CIred_edge)
- Water content (NDWI, MSI, NMDI, LSWI)
- Soil salinity (SI, BI, SI1, SI2, SI3, SRSI)
- Crop stress detection

Data sources: Sentinel-2 (10m/20m), EnMAP (30m), Planet Tanager (20m)

In [ ]:
!pip -q install rasterio rioxarray xarray numpy matplotlib scipy

In [ ]:
import numpy as np
import xarray as xr
import rioxarray
import matplotlib.pyplot as plt
from pathlib import Path

# Sentinel-2 band mapping (L2A)
SENTINEL2_BANDS = {
    'B01': 443,
    'B02': 490,
    'B03': 560,
    'B04': 665,
    'B05': 705,
    'B06': 740,
    'B07': 783,
    'B08': 842,
    'B8A': 865,
    'B09': 945,
    'B11': 1610,
    'B12': 2190,
}

def find_band_idx(wavelengths, target_nm):
    """Find closest band index to target wavelength."""
    return np.argmin(np.abs(wavelengths - target_nm))

## 1. Vegetation Indices

In [ ]:
def compute_vegetation_indices(cube, wavelengths=None, band_names=None):
    """
    Compute vegetation indices from Sentinel-2 or hyperspectral data.
    
    For Sentinel-2: pass band_names dict with keys like 'B04', 'B08', etc.
    For hyperspectral: pass wavelengths array
    """
    indices = {}
    
    if band_names is not None:
        B02 = cube.sel(band='B02') if 'band' in cube.dims else cube[0]
        B03 = cube.sel(band='B03') if 'band' in cube.dims else cube[1]
        B04 = cube.sel(band='B04') if 'band' in cube.dims else cube[2]
        B05 = cube.sel(band='B05') if 'band' in cube.dims else cube[3]
        B06 = cube.sel(band='B06') if 'band' in cube.dims else cube[4]
        B07 = cube.sel(band='B07') if 'band' in cube.dims else cube[5]
        B08 = cube.sel(band='B08') if 'band' in cube.dims else cube[6]
        B8A = cube.sel(band='B8A') if 'band' in cube.dims else cube[7]
        B11 = cube.sel(band='B11') if 'band' in cube.dims else cube[8]
        B12 = cube.sel(band='B12') if 'band' in cube.dims else cube[9]
        
        eps = 1e-6
        
        indices['NDVI'] = (B08 - B04) / (B08 + B04 + eps)
        indices['EVI'] = 2.5 * (B08 - B04) / (B08 + 6*B04 - 7.5*B02 + 1 + eps)
        L = 0.5
        indices['SAVI'] = (B08 - B04) / (B08 + B04 + L) * (1 + L)
        indices['NDRE'] = (B08 - B05) / (B08 + B05 + eps)
        indices['NDRE2'] = (B08 - B06) / (B08 + B06 + eps)
        indices['CIred_edge'] = B08 / B05 - 1
        indices['GNDVI'] = (B08 - B03) / (B08 + B03 + eps)
        indices['REIP'] = 700 + 40 * ((B04 + B07)/2 - B05) / (B06 - B05 + eps)
    elif wavelengths is not None:
        pass
    
    return indices

## 2. Water Content Indices

In [ ]:
def compute_water_indices(cube, band_names=None):
    """Compute water content indices."""
    indices = {}
    
    if band_names is not None:
        B03 = cube.sel(band='B03')
        B08 = cube.sel(band='B08')
        B11 = cube.sel(band='B11')
        B12 = cube.sel(band='B12')
        B8A = cube.sel(band='B8A')
        
        eps = 1e-6
        
        indices['NDWI'] = (B03 - B08) / (B03 + B08 + eps)
        indices['NDWI_Gao'] = (B08 - B11) / (B08 + B11 + eps)
        indices['MSI'] = B11 / B08
        indices['NMDI'] = (B08 - (B11 - B12)) / (B08 + (B11 - B12) + eps)
        indices['LSWI'] = (B08 - B11) / (B08 + B11 + eps)
        indices['MNDWI'] = (B03 - B11) / (B03 + B11 + eps)
    
    return indices

## 3. Soil Salinity Indices (Critical for Liwa)

In [ ]:
def compute_salinity_indices(cube, band_names=None):
    """Compute soil salinity indices - KEY for Liwa desert agriculture."""
    indices = {}
    
    if band_names is not None:
        B04 = cube.sel(band='B04')
        B08 = cube.sel(band='B08')
        B11 = cube.sel(band='B11')
        B12 = cube.sel(band='B12')
        B03 = cube.sel(band='B03')
        B02 = cube.sel(band='B02')
        
        eps = 1e-6
        
        indices['SI'] = (B11 - B04) / (B11 + B04 + eps)
        indices['BI'] = np.sqrt(B11**2 + B04**2)
        indices['SI1'] = B11 / B08
        indices['SI2'] = (B12 - B04) / (B12 + B04 + eps)
        indices['SI3'] = (B12 - B11) / (B12 + B11 + eps)
        indices['SRSI'] = B11 / B03
        indices['CSRI'] = (B11 / B08) * (1 - (B08 - B04) / (B08 + B04 + eps))
    
    return indices

## 4. Complete Index Computation Pipeline

In [ ]:
def compute_all_indices(cube, band_names=None, wavelengths=None):
    """Compute all indices in one call."""
    all_indices = {}
    
    if band_names is not None:
        all_indices.update(compute_vegetation_indices(cube, band_names=band_names))
        all_indices.update(compute_water_indices(cube, band_names=band_names))
        all_indices.update(compute_salinity_indices(cube, band_names=band_names))
    
    return all_indices


def save_indices_as_cog(indices, reference_raster, output_dir):
    """Save computed indices as Cloud-Optimized GeoTIFFs."""
    import rasterio
    from rasterio.enums import Resampling
    
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    with rasterio.open(reference_raster) as ref:
        profile = ref.profile.copy()
        profile.update(
            driver='COG',
            compress='deflate',
            dtype='float32',
            count=1
        )
        
        for name, index_data in indices.items():
            data = index_data.astype(np.float32)
            data = np.nan_to_num(data, nan=-9999, posinf=-9999, neginf=-9999)
            
            output_path = output_dir / f"{name}.tif"
            with rasterio.open(output_path, 'w', **profile) as dst:
                dst.write(data, 1)
                dst.set_band_description(1, name)
            
            print(f"Saved: {output_path}")

## 5. Visualization & Interpretation

In [ ]:
def plot_index_map(index_data, title, cmap='RdYlGn', vmin=None, vmax=None):
    """Plot a single index map."""
    plt.figure(figsize=(10, 8))
    im = plt.imshow(index_data, cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(im, label=title)
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()


def plot_indices_grid(indices_dict, ncols=3, figsize=(15, 10)):
    """Plot multiple indices in a grid."""
    n = len(indices_dict)
    nrows = (n + ncols - 1) // ncols
    
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = axes.flatten() if n > 1 else [axes]
    
    for ax, (name, data) in zip(axes, indices_dict.items()):
        if 'NDVI' in name or 'EVI' in name or 'SAVI' in name or 'NDRE' in name:
            cmap = 'RdYlGn'
            vmin, vmax = -0.2, 1.0
        elif 'NDWI' in name or 'MSI' in name or 'NMDI' in name or 'LSWI' in name:
            cmap = 'BrBG'
            vmin, vmax = -1, 1
        elif 'SI' in name or 'BI' in name or 'SRSI' in name or 'CSRI' in name:
            cmap = 'RdBu_r'
            vmin, vmax = -1, 1
        else:
            cmap = 'viridis'
            vmin, vmax = None, None
        
        im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(name)
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    
    for ax in axes[n:]:
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

## 6. Thresholds for Classification

In [ ]:
# Recommended thresholds for Liwa desert agriculture context

THRESHOLDS = {
    'NDVI': {'bare': 0.1, 'sparse': 0.2, 'moderate': 0.4, 'dense': 0.6},
    'EVI': {'bare': 0.05, 'sparse': 0.15, 'moderate': 0.3, 'dense': 0.5},
    'SAVI': {'bare': 0.1, 'sparse': 0.2, 'moderate': 0.35, 'dense': 0.5},
    'NDWI': {'water': 0.3, 'wet': 0.1, 'dry': -0.1, 'stressed': -0.3},
    'MSI': {'low_stress': 0.5, 'moderate': 1.0, 'high_stress': 1.5},
    'NMDI': {'wet': 0.3, 'normal': 0.0, 'dry': -0.3, 'severe': -0.5},
    'SI': {'low': 0.0, 'moderate': 0.2, 'high': 0.4, 'severe': 0.6},
    'SI3': {'low': -0.2, 'moderate': 0.0, 'high': 0.2, 'severe': 0.4},
    'BI': {'dark': 0.1, 'normal': 0.3, 'bright': 0.5, 'very_bright': 0.7},
}

def classify_index(index_data, index_name, thresholds=THRESHOLDS):
    """Classify index values into categories."""
    if index_name not in thresholds:
        return None
    
    thresh = thresholds[index_name]
    classes = np.zeros_like(index_data, dtype=np.uint8)
    
    if 'NDVI' in index_name or 'EVI' in index_name or 'SAVI' in index_name:
        classes[index_data < thresh['bare']] = 0
        classes[(index_data >= thresh['bare']) & (index_data < thresh['sparse'])] = 1
        classes[(index_data >= thresh['sparse']) & (index_data < thresh['moderate'])] = 2
        classes[(index_data >= thresh['moderate']) & (index_data < thresh['dense'])] = 3
        classes[index_data >= thresh['dense']] = 4
    elif 'NDWI' in index_name:
        classes[index_data >= thresh['water']] = 0
        classes[(index_data < thresh['water']) & (index_data >= thresh['wet'])] = 1
        classes[(index_data < thresh['wet']) & (index_data >= thresh['dry'])] = 2
        classes[(index_data < thresh['dry']) & (index_data >= thresh['stressed'])] = 3
        classes[index_data < thresh['stressed']] = 4
    
    return classes